In [0]:
from pyspark.sql.functions import col, current_timestamp

# Read from silver layer
df_silver = spark.table("ecommerce.e_comm_silver.regions")
df_silver.createOrReplaceTempView("vw_silver")
# Transform for gold layer:
# 1. Filter only valid records
# 2. Select business columns only
# 3. Rename for business clarity
df_gold_dim_regions = spark.sql("""
        select 
            region_key,
            region_id,
            region_name,
            current_timestamp() as load_ts
        from vw_silver
        where dq_note ="is_valid"
    """)

# Display sample
print(f"Total records in gold dimension: {df_gold_dim_regions.count()}")

# Write to gold layer
df_gold_dim_regions.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce.e_comm_gold.dimRegions")

print("✅ Gold dimension table ecommerce.e_comm_gold.dimRegions created successfully")